Event collection / streaming semantics — we have Kafka, but not a realistic ingestion layer, event-time/processing-time semantics, partitions, offsets, consumer lag, delivery semantics, etc.
Feature engineering — Flink/Spark compute features, but the feature definitions themselves aren't yet modeled as proper reusable feature views/entities with point-in-time semantics.
Training pipeline — we have the basic DataLake → Spark → feature store → dataset → trainer flow, but not realistic dataset construction, temporal joins, negatives, splits, leakage prevention, or retraining lifecycle.
Model lifecycle — registry exists, but model/version deployment, validation, rollback, and serving updates are rudimentary.
Retrieval/serving — Two-Tower + HNSW works, but production retrieval needs realistic candidate-generation behavior, embedding/index lifecycle, latency, filtering, and serving concerns.
System dynamics — latency, throughput, backpressure, failures, stale features, skew, bursts, hot keys, retries, lag, and resource constraints are largely absent.
Architecture fidelity — several components currently have the names of production systems without yet modeling the important behavior those systems provide.

To do next
Verify impression → click linkage.
Replace random negatives with actual impression-based negatives.
Fix temporal leakage in negative construction.
Re-evaluate Two-Tower vs popularity.
If still weak, investigate:
feature normalization
item-ID embedding / memorization
click vs watch label design
Later: make Feast, Spark, Flink, etc. progressively more production-realistic.
Eventually turn components into reusable Lego-style simulation blocks for different ML systems.

training objective — BCE on individual positive/negative pairs may not be the right objective for two-tower retrieval;
feature representation — especially the user/item feature construction;
feature scaling/normalization;
item-ID embedding and its effect on generalization;
train/evaluation mismatch — impression negatives during training vs full-catalog retrieval during evaluation;
optimization — learning rate, epochs, initialization, etc.

The retriever needs these changes:

Normalize user features
genre preferences
rating
activity
watch count
click count
interactions
Normalize item features
rating
popularity
Normalize the final tower embeddings so dot-product retrieval is effectively cosine similarity.
Keep the movie-ID embedding for now. We already identified its memorization/generalization issue, but that's a separate architectural decision. Removing it now would change too many things at once.

So the immediate fix is feature normalization + L2-normalized output embeddings, on both towers.

And importantly: because we're changing .py, we'll edit the actual two_tower.py and reload it in the same cell. No simulator rerun until the code is changed.

what the next experiment is: batched InfoNCE with in-batch negatives.

In [1]:
!git --version

git version 2.43.0


In [2]:
from getpass import getpass

token = getpass("GitHub token: ")

GitHub token: ··········


In [3]:
import os

os.environ["GITHUB_TOKEN"] = token

In [4]:
!git clone https://$GITHUB_TOKEN@github.com/PrimozRavbar/SimPipelineExp.git

Cloning into 'SimPipelineExp'...
remote: Enumerating objects: 94, done.
remote: Counting objects: 100% (94/94), done.
remote: Compressing objects: 100% (90/90), done.
remote: Total 94 (delta 19), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (94/94), 8.07 MiB | 9.53 MiB/s, done.
Resolving deltas: 100% (19/19), done.


In [8]:
!git config user.name "PrimozRavbar"
!git config user.email "96559695+PrimozRavbar@users.noreply.github.com"

In [ ]:
"""
# commit to GH !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

!git config user.name "PrimozRavbar"
!git config user.email "96559695+PrimozRavbar@users.noreply.github.com"

!git add .
!git commit -m "improved display of simulation"
!git push

[main 1e20ba1] improved display of simulation
 4 files changed, 23 insertions(+), 9 deletions(-)
Enumerating objects: 17, done.
Counting objects: 100% (17/17), done.
Delta compression using up to 2 threads
Compressing objects: 100% (9/9), done.
Writing objects: 100% (9/9), 1012 bytes | 1012.00 KiB/s, done.
Total 9 (delta 8), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (8/8), completed with 8 local objects.
To https://github.com/PrimozRavbar/SimPipeline.git
   8a0b782..1e20ba1  main -> main


In [6]:
%cd /content/SimPipelineExp
!git status

/content/SimPipelineExp
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [7]:
%cd /content/SimPipelineExp
!git pull

/content/SimPipelineExp
Already up to date.


In [9]:

from world.external_data_loader import load_movielens

world_data = load_movielens("/content/SimPipelineExp/ml-100k")

print("Loader OK:", len(world_data["user_objects"]), len(world_data["movie_objects"]), len(world_data["rating_objects"]))

/content/SimPipelineExp/world/external_data_loader.py:27: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  ratings = pd.read_csv(


Loader OK: 943 1682 100000


In [10]:
import importlib

import ml_system.datalake
import ml_system.training_dataset
import ml_system.evaluation
import ml_system.build_recommendation_system


from ml_system.build_recommendation_system import build_recommendation_system
from ml_system.evaluation import RetrievalEvaluator, evaluate_popularity

In [11]:
import importlib

import simulation.simulator
import world.world
import world.user_simulator

importlib.reload(simulation.simulator)
importlib.reload(world.user_simulator)
importlib.reload(world.world)


module = importlib.import_module(build_recommendation_system.__module__)
importlib.reload(module)

build_recommendation_system = module.build_recommendation_system

In [12]:
from ml_system.build_recommendation_system import build_recommendation_system

system = build_recommendation_system(
    movie_objects=world_data["movie_objects"],
    user_objects=world_data["user_objects"],
    user_states=world_data["user_states"],
    movie_states=world_data["movie_states"],
)

print("System build OK")

System build OK


In [13]:
system["datalake"].events.clear()
print("DataLake events:", len(system["datalake"].events))

DataLake events: 0


In [14]:
system["sim"].run(5000)

Steps: 5000 | Events: 10723 | RecommendationShownEvent: 5000 | RecommendationClickedEvent: 3053 | WatchEvent: 2437 | RatingEvent: 233


In [15]:
#system["sim"].run(5000)

events = system["datalake"].events

from collections import Counter

counts = Counter(type(event).__name__ for event in events)

print("Total events:", len(events))
print(counts)

Total events: 10723
Counter({'RecommendationShownEvent': 5000, 'RecommendationClickedEvent': 3053, 'WatchEvent': 2437, 'RatingEvent': 233})


In [14]:
print("DataLake events:", len(system["datalake"].events))

DataLake events: 10564


In [16]:
system["spark"].run_offline_feature_job(
    system["datalake"].events,
    system["offline_pipeline"].feature_pipeline
)
print("Spark offline feature job complete.")

Spark offline feature job complete.


In [17]:
print("Application:", system["spark"].application.state)
print("Application metrics:", system["spark"].application.metrics)
print("Driver:", system["spark"].application.driver.state)
print("Job:", system["spark"].application.jobs[0].state)
print("Stage:", system["spark"].application.jobs[0].stages[0].state)
print("Task:", system["spark"].application.jobs[0].stages[0].tasks[0].state)
print("Executor:", system["spark"].executor.state)

Application: ApplicationState.SUCCEEDED
Application metrics: {'jobs_submitted': 1, 'jobs_succeeded': 1, 'jobs_failed': 0}
Driver: DriverState.COMPLETED
Job: JobState.SUCCEEDED
Stage: StageState.SUCCEEDED
Task: TaskState.SUCCEEDED
Executor: ExecutorState.IDLE


In [ ]:
from tests.test_online_offline_features import test_online_offline_feature_parity

test_online_offline_feature_parity(system)

print("Feature parity OK")

AssertionError: 

In [18]:
from ml_system.retrain import retrain

system["offline_pipeline"].dataset_generator.negative_samples = 5
system["offline_pipeline"].trainer.epochs = 5

artifacts = retrain(system)

epoch: 1 loss: 3.6243403578797975
epoch: 2 loss: 3.555814671019713
epoch: 3 loss: 3.475610204041004
epoch: 4 loss: 3.3436903233329454
epoch: 5 loss: 3.1919713417689004


In [19]:
from simulation.events import RecommendationClickedEvent, WatchEvent
from simulation.events import RetrievalRequest

events = system["datalake"].events

hits = 0
evaluated = 0

for event in events:
    if not isinstance(event, (RecommendationClickedEvent, WatchEvent)):
        continue

    recommendations = system["retriever"].retrieve(
        RetrievalRequest(
            user_id=event.user_id,
            k=20
        )
    )

    evaluated += 1

    if event.movie_id in recommendations:
        hits += 1

recall_at_20 = hits / evaluated if evaluated else 0.0

print(f"Hits: {hits}")
print(f"Evaluated: {evaluated}")
print(f"Recall@20: {recall_at_20:.4%}")

Hits: 916
Evaluated: 5490
Recall@20: 16.6849%


For a two-tower retrieval model, let's replace the independent BCE-on-dot-product objective with a sampled softmax / InfoNCE-style contrastive loss:

$$ L=-\log\frac{e^{s^+/\tau}} {e^{s^+/\tau}+\sum_j e^{s_j^-/\tau}} $$

That means every impression only competes against its own 5 negatives. We're throwing away a huge source of negatives that contrastive two-tower systems commonly exploit: other positives in the batch.

That is the next change I'd make: mini-batch InfoNCE with in-batch negatives.